In [1]:
!pip -q install "surya-ocr==0.17.1" pymupdf opencv-python-headless pandas openpyxl pillow jiwer

In [2]:
!pip install -q --force-reinstall "transformers==4.57.3"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
numba 0.64.0 requires numpy<2.5,>=1.22, but you have numpy 2.5.4 which is incompatible.
datasets 4.8.5 requires fsspec[http]<=2026.2.0,>=2023.1.0, but you have fsspec 2026.9.0 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2026.9.0 which is incompatible.


In [3]:
import transformers
print(transformers.__version__)

4.57.3


In [4]:
import os
import re
import json
import math
import hashlib
import warnings

import cv2
import pymupdf as fitz
import torch
import numpy as np
import pandas as pd

from pathlib import Path
from PIL import Image
from IPython.display import display

from surya.foundation import FoundationPredictor
from surya.recognition import RecognitionPredictor
from surya.detection import DetectionPredictor

warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)
print("GPU available:", torch.cuda.is_available())

foundation_predictor = FoundationPredictor()

recognition_predictor = RecognitionPredictor(
    foundation_predictor
)

detection_predictor = DetectionPredictor()

print("Surya OCR initialized successfully.")

Device: cuda
GPU available: True


Surya OCR initialized successfully.


In [6]:
from google.colab import drive

drive.mount("/content/drive")

INPUT_DIR = Path("/content/drive/MyDrive/handwritten")
OUTPUT_DIR = Path("/content/handwritten_ocr/outputs")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DPI = 200

pdf_files = sorted(INPUT_DIR.rglob("*.pdf"))

print("PDF files found:", len(pdf_files))

for pdf in pdf_files:
    print(pdf.name)

if not pdf_files:
    raise FileNotFoundError(
        f"No PDF files found in {INPUT_DIR}"
    )

Mounted at /content/drive
PDF files found: 9
biodiversity and conservation.pdf
human eye and colorful world.pdf
microbes in human welfare.pdf
minerals and energy resources.pdf
ml notes -lalit preparation.pdf
money and credit.pdf
organisms and populations.pdf
political parties.pdf
rise of nationalism in europe.pdf


In [7]:
def render_page(page, dpi=200):

    scale = dpi / 72

    pix = page.get_pixmap(
        matrix=fitz.Matrix(scale, scale),
        alpha=False
    )

    image = np.frombuffer(
        pix.samples,
        dtype=np.uint8
    ).reshape(pix.height, pix.width, 3)

    return image.copy()


def estimate_skew(image):

    gray = cv2.cvtColor(
        image, cv2.COLOR_RGB2GRAY
    )

    edges = cv2.Canny(gray, 50, 150)

    lines = cv2.HoughLinesP(
        edges,
        1,
        np.pi / 180,
        threshold=100,
        minLineLength=max(40, image.shape[1] // 5),
        maxLineGap=20
    )

    if lines is None:
        return 0.0

    angles = []

    for line in lines:
        x1, y1, x2, y2 = line[0]

        angle = np.degrees(
            np.arctan2(y2 - y1, x2 - x1)
        )

        if -15 <= angle <= 15:
            angles.append(angle)

    if not angles:
        return 0.0

    return float(np.median(angles))


def preprocess_page(image):

    original = image.copy()

    skew_before = estimate_skew(original)

    if abs(skew_before) > 0.5:

        height, width = original.shape[:2]

        center = (width // 2, height // 2)

        matrix = cv2.getRotationMatrix2D(
            center,
            -skew_before,
            1.0
        )

        corrected = cv2.warpAffine(
            original,
            matrix,
            (width, height),
            flags=cv2.INTER_CUBIC,
            borderMode=cv2.BORDER_REPLICATE
        )

    else:
        corrected = original.copy()

    denoised = cv2.fastNlMeansDenoisingColored(
        corrected,
        None,
        5,
        5,
        7,
        21
    )

    lab = cv2.cvtColor(
        denoised,
        cv2.COLOR_RGB2LAB
    )

    l_channel, a_channel, b_channel = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )

    enhanced_l = clahe.apply(l_channel)

    enhanced = cv2.merge([
        enhanced_l,
        a_channel,
        b_channel
    ])

    processed = cv2.cvtColor(
        enhanced,
        cv2.COLOR_LAB2RGB
    )

    skew_after = estimate_skew(processed)

    return processed, skew_before, skew_after


def get_image_metrics(image):

    gray = cv2.cvtColor(
        image,
        cv2.COLOR_RGB2GRAY
    )

    height, width = gray.shape

    blur = float(
        cv2.Laplacian(
            gray,
            cv2.CV_64F
        ).var()
    )

    contrast = float(gray.std())

    return {
        "width": width,
        "height": height,
        "resolution": width * height,
        "blur_score": blur,
        "contrast": contrast
    }

In [8]:
def clean_text(text):

    text = str(text)

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


def run_surya_ocr(image):

    pil_image = Image.fromarray(
        image.astype(np.uint8)
    ).convert("RGB")

    predictions = recognition_predictor(
    [pil_image],
    det_predictor=detection_predictor
)

    if not predictions:
        return []

    prediction = predictions[0]

    results = []

    for line in prediction.text_lines:

        text = clean_text(line.text)

        if not text:
            continue

        bbox = getattr(line, "bbox", None)
        polygon = getattr(line, "polygon", None)

        if bbox is None and polygon is not None:
            points = np.array(polygon)

            bbox = [
                points[:, 0].min(),
                points[:, 1].min(),
                points[:, 0].max(),
                points[:, 1].max()
            ]

        if bbox is None:
            continue

        confidence = getattr(
            line,
            "confidence",
            None
        )

        results.append({
            "text": text,
            "confidence": (
                float(confidence)
                if confidence is not None
                else np.nan
            ),
            "bbox": [
                float(value)
                for value in bbox
            ],
            "polygon": (
                polygon.tolist()
                if hasattr(polygon, "tolist")
                else polygon
            )
        })

    results.sort(
        key=lambda item: (
            item["bbox"][1],
            item["bbox"][0]
        )
    )

    return results


def combine_ocr_text(results):

    return "\n".join(
        item["text"]
        for item in results
    )


def average_confidence(results):

    values = [
        item["confidence"]
        for item in results
        if pd.notna(item["confidence"])
    ]

    if not values:
        return np.nan

    return float(np.mean(values))

In [9]:
def detect_table(image):

    gray = cv2.cvtColor(
        image,
        cv2.COLOR_RGB2GRAY
    )

    binary = cv2.adaptiveThreshold(
        gray,
        255,
        cv2.ADAPTIVE_THRESH_MEAN_C,
        cv2.THRESH_BINARY_INV,
        31,
        15
    )

    height, width = binary.shape

    horizontal_kernel = cv2.getStructuringElement(
        cv2.MORPH_RECT,
        (max(30, width // 10), 1)
    )

    vertical_kernel = cv2.getStructuringElement(
        cv2.MORPH_RECT,
        (1, max(30, height // 10))
    )

    horizontal = cv2.morphologyEx(
        binary,
        cv2.MORPH_OPEN,
        horizontal_kernel
    )

    vertical = cv2.morphologyEx(
        binary,
        cv2.MORPH_OPEN,
        vertical_kernel
    )

    intersections = cv2.bitwise_and(
        horizontal,
        vertical
    )

    return (
        cv2.countNonZero(horizontal) > width * 0.5
        and cv2.countNonZero(vertical) > height * 0.1
        and cv2.countNonZero(intersections) > 20
    )


def detect_equations(text):

    pattern = (
        r"[=+\-×÷√∑∫<>]"
        r"|\b(sin|cos|tan|log|lim|frac)\b"
        r"|\d+\s*/\s*\d+"
    )

    return bool(
        re.search(pattern, text, re.IGNORECASE)
    )


def detect_diagram(image):

    gray = cv2.cvtColor(
        image,
        cv2.COLOR_RGB2GRAY
    )

    edges = cv2.Canny(
        gray,
        50,
        150
    )

    edge_ratio = (
        cv2.countNonZero(edges)
        / max(1, gray.size)
    )

    return edge_ratio > 0.08


def classify_page(
    confidence,
    metrics,
    skew,
    handwriting_suspected,
    equation,
    table,
    diagram,
    text
):

    score = 0
    reasons = []

    if pd.isna(confidence):
        score += 2
        reasons.append("Missing OCR confidence")
    elif confidence < 0.50:
        score += 2
        reasons.append("Low OCR confidence")
    elif confidence < 0.75:
        score += 1
        reasons.append("Moderate OCR confidence")

    if metrics["blur_score"] < 35:
        score += 2
        reasons.append("Blurry page")
    elif metrics["blur_score"] < 80:
        score += 1
        reasons.append("Moderate blur")

    short_side = min(
        metrics["width"],
        metrics["height"]
    )

    if short_side < 900:
        score += 2
        reasons.append("Low resolution")
    elif short_side < 1200:
        score += 1
        reasons.append("Moderate resolution")

    if abs(skew) > 3:
        score += 2
        reasons.append("High skew")
    elif abs(skew) > 1:
        score += 1
        reasons.append("Moderate skew")

    if handwriting_suspected:
        score += 2
        reasons.append("Handwriting suspected")

    if equation:
        score += 1
        reasons.append("Equation detected")

    if table:
        score += 1
        reasons.append("Table detected")

    if diagram:
        score += 1
        reasons.append("Diagram/visual complexity suspected")

    if len(text.strip()) < 30:
        score += 1
        reasons.append("Very little recognized text")

    if score <= 2:
        category = "Easy"
    elif score <= 5:
        category = "Moderate"
    else:
        category = "Difficult"

    return category, score, reasons

In [10]:
page_records = []
chunk_records = []

document_counter = 0

for pdf_path in pdf_files:

    document_counter += 1
    document_id = f"DOC_{document_counter:04d}"

    print("\nProcessing:", pdf_path.name)

    pdf = fitz.open(pdf_path)

    for page_index in range(len(pdf)):

        page_number = page_index + 1

        page_id = f"{document_id}_P{page_number:04d}"

        print(f"  Processing page {page_number}/{len(pdf)}")

        original_image = render_page(
            pdf[page_index],
            DPI
        )

        processed_image, skew_before, skew_after = (
            preprocess_page(original_image)
        )

        original_metrics = get_image_metrics(original_image)
        processed_metrics = get_image_metrics(processed_image)

        original_results = run_surya_ocr(original_image)
        processed_results = run_surya_ocr(processed_image)

        original_text = combine_ocr_text(original_results)
        processed_text = combine_ocr_text(processed_results)

        original_confidence = average_confidence(original_results)
        processed_confidence = average_confidence(processed_results)

        equation = detect_equations(processed_text)

        table = detect_table(original_image)
        diagram = detect_diagram(original_image)

        handwriting_suspected = False

        category, difficulty_score, reasons = classify_page(
            processed_confidence,
            processed_metrics,
            skew_before,
            handwriting_suspected,
            equation,
            table,
            diagram,
            processed_text
        )

        original_char_count = len(original_text)
        processed_char_count = len(processed_text)

        char_difference = (
            processed_char_count - original_char_count
        )

        if original_char_count > 0:
            char_change_percent = (
                char_difference / original_char_count
            ) * 100
        else:
            char_change_percent = np.nan

        page_records.append({

            "document_id": document_id,
            "document_name": pdf_path.name,
            "page_id": page_id,
            "page_number": page_number,

            "original_text": original_text,
            "preprocessed_text": processed_text,

            "original_confidence": original_confidence,
            "preprocessed_confidence": processed_confidence,

            "original_line_count": len(original_results),
            "preprocessed_line_count": len(processed_results),

            "original_char_count": original_char_count,
            "preprocessed_char_count": processed_char_count,

            "char_difference": char_difference,
            "char_change_percent": char_change_percent,

            "width": processed_metrics["width"],
            "height": processed_metrics["height"],

            "original_blur": original_metrics["blur_score"],
            "processed_blur": processed_metrics["blur_score"],

            "original_contrast": original_metrics["contrast"],
            "processed_contrast": processed_metrics["contrast"],

            "skew_before": skew_before,
            "skew_after": skew_after,

            "handwriting_suspected": handwriting_suspected,
            "equation_detected": equation,
            "table_detected": table,
            "diagram_suspected": diagram,

            "difficulty_score": difficulty_score,
            "processing_category": category,
            "difficulty_reasons": "; ".join(reasons),

            "review_required": (
                category == "Difficult"
                or pd.isna(processed_confidence)
                or processed_confidence < 0.60
            )
        })

        for line_number, item in enumerate(original_results, start=1):

            chunk_id = f"{page_id}_ORIG_C{line_number:04d}"

            chunk_records.append({

                "document_id": document_id,
                "page_id": page_id,
                "chunk_id": chunk_id,

                "document_name": pdf_path.name,
                "page_number": page_number,

                "variant": "Original",
                "line_number": line_number,

                "text": item["text"],
                "confidence": item["confidence"],
                "bbox": json.dumps(item["bbox"]),
                "polygon": json.dumps(item["polygon"])
            })

        for line_number, item in enumerate(processed_results, start=1):

            chunk_id = f"{page_id}_PRE_C{line_number:04d}"

            chunk_records.append({

                "document_id": document_id,
                "page_id": page_id,
                "chunk_id": chunk_id,

                "document_name": pdf_path.name,
                "page_number": page_number,

                "variant": "Preprocessed",
                "line_number": line_number,

                "text": item["text"],
                "confidence": item["confidence"],
                "bbox": json.dumps(item["bbox"]),
                "polygon": json.dumps(item["polygon"])
            })

    pdf.close()

pages_df = pd.DataFrame(page_records)
chunks_df = pd.DataFrame(chunk_records)

print("\nProcessing completed.")

print("Documents:", pages_df["document_id"].nunique())
print("Pages:", len(pages_df))
print("Chunks:", len(chunks_df))

display(
    pages_df[
        [
            "document_id",
            "page_id",
            "original_confidence",
            "preprocessed_confidence",
            "processing_category",
            "review_required"
        ]
    ].head()
)


Processing: biodiversity and conservation.pdf
  Processing page 1/13


Recognizing Text: 100%|██████████| 14/14 [00:05<00:00,  2.70it/s]


  Processing page 2/13


Recognizing Text: 100%|██████████| 23/23 [00:02<00:00,  8.77it/s]


  Processing page 3/13


Recognizing Text: 100%|██████████| 24/24 [00:03<00:00,  6.02it/s]


... (270 outputs left out)

  Processing page 17/17


Recognizing Text: 100%|██████████| 35/35 [00:01<00:00, 18.57it/s]



Processing completed.
Documents: 9
Pages: 139
Chunks: 9566


,document_id,page_id,original_confidence,preprocessed_confidence,processing_category,review_required
0,DOC_0001,DOC_0001_P0001,0.971818,0.924539,Easy,False
1,DOC_0001,DOC_0001_P0002,0.897507,0.904472,Easy,False
2,DOC_0001,DOC_0001_P0003,0.900289,0.912973,Moderate,False
3,DOC_0001,DOC_0001_P0004,0.931424,0.952276,Easy,False
4,DOC_0001,DOC_0001_P0005,0.969743,0.975166,Easy,False


In [11]:
comparison_df = pages_df[
    [
        "document_id",
        "document_name",
        "page_id",
        "page_number",

        "original_text",
        "preprocessed_text",

        "original_confidence",
        "preprocessed_confidence",

        "original_line_count",
        "preprocessed_line_count",

        "original_char_count",
        "preprocessed_char_count",

        "char_difference",
        "char_change_percent"
    ]
].copy()

comparison_df["confidence_difference"] = (
    comparison_df["preprocessed_confidence"]
    - comparison_df["original_confidence"]
)

comparison_df["confidence_improved"] = (
    comparison_df["confidence_difference"] > 0
)

comparison_df["text_changed"] = (
    comparison_df["original_text"]
    != comparison_df["preprocessed_text"]
)

display(
    comparison_df[
        [
            "page_id",
            "original_confidence",
            "preprocessed_confidence",
            "confidence_difference",
            "text_changed"
        ]
    ].head(20)
)

,page_id,original_confidence,preprocessed_confidence,confidence_difference,text_changed
0,DOC_0001_P0001,0.971818,0.924539,-0.047279,True
1,DOC_0001_P0002,0.897507,0.904472,0.006965,True
2,DOC_0001_P0003,0.900289,0.912973,0.012684,True
3,DOC_0001_P0004,0.931424,0.952276,0.020852,True
4,DOC_0001_P0005,0.969743,0.975166,0.005423,True
5,DOC_0001_P0006,0.972619,0.945872,-0.026747,True
6,DOC_0001_P0007,0.978000,0.971822,-0.006178,True
7,DOC_0001_P0008,0.933464,0.920088,-0.013375,True
8,DOC_0001_P0009,0.969861,0.959456,-0.010405,True
9,DOC_0001_P0010,0.935025,0.920348,-0.014677,True


In [12]:
def show_ocr_examples(data, number=5):

    for _, row in data.head(number).iterrows():

        print("=" * 80)

        print("Document:", row["document_name"])
        print("Page ID:", row["page_id"])

        print("\nORIGINAL OCR:")
        print(row["original_text"])

        print("\nPREPROCESSED OCR:")
        print(row["preprocessed_text"])

        print("\nOriginal confidence:",
              row["original_confidence"])

        print("Preprocessed confidence:",
              row["preprocessed_confidence"])

        print("\n")


show_ocr_examples(
    comparison_df,
    number=5
)

Document: biodiversity and conservation.pdf
Page ID: DOC_0001_P0001

ORIGINAL OCR:
diversitu
nservation
According to sociobiologist Edward wilson -
ee The combined diversity at all levels of
biological auganisation is known as Biodi-
The most impartant of them are -
1. Menetic Diversity - Genetic diversity refers
to the different types of genes that are present in a population are in a given species. Frample: Rauwolfia vomitoria
growing in different Himalayan ranges have different in term of Potency and
concentuation of active chemical (Hesper-
pine) which is used in Blood phessure
treatment.
2. Species Diversity — Diversity at the species level is species diversity. Fxample: The western Chats have greater amphbian species diversity than the estern ghats.
3. Ecological Diversity — Fcological diversity is the variety of life farms that exist in different ecosystems.

PREPROCESSED OCR:
diversity
nservation
According to sociobiologist Edward wilson -
hiological auganisation is known as Bi

In [13]:
error_records = []

for _, row in pages_df.iterrows():

    potential_sources = []

    if row["handwriting_suspected"]:
        potential_sources.append("Handwriting")

    if min(row["width"], row["height"]) < 1200:
        potential_sources.append("Low resolution")

    if abs(row["skew_before"]) > 1:
        potential_sources.append("Skew")

    if row["equation_detected"]:
        potential_sources.append("Equations")

    if row["table_detected"]:
        potential_sources.append("Tables")

    if row["diagram_suspected"]:
        potential_sources.append("Diagrams")

    if (
        pd.notna(row["preprocessed_confidence"])
        and row["preprocessed_confidence"] < 0.60
    ):
        potential_sources.append("Low OCR confidence")

    if not row["preprocessed_text"].strip():
        potential_sources.append("No recognized text")

    error_records.append({

        "document_id": row["document_id"],
        "document_name": row["document_name"],
        "page_id": row["page_id"],
        "page_number": row["page_number"],

        "potential_error_sources": "; ".join(
            potential_sources
        ),

        "handwriting_suspected": row["handwriting_suspected"],
        "low_resolution": (
            min(row["width"], row["height"]) < 1200
        ),
        "skew_detected": abs(row["skew_before"]) > 1,

        "equation_detected": row["equation_detected"],
        "table_detected": row["table_detected"],
        "diagram_suspected": row["diagram_suspected"],

        "ocr_confidence": row["preprocessed_confidence"],

    })

error_df = pd.DataFrame(error_records)

display(error_df.head(20))

,document_id,document_name,page_id,page_number,potential_error_sources,handwriting_suspected,low_resolution,skew_detected,equation_detected,table_detected,diagram_suspected,ocr_confidence
0,DOC_0001,biodiversity and conservation.pdf,DOC_0001_P0001,1,Equations; Tables,False,False,False,True,True,False,0.924539
1,DOC_0001,biodiversity and conservation.pdf,DOC_0001_P0002,2,Equations; Tables,False,False,False,True,True,False,0.904472
2,DOC_0001,biodiversity and conservation.pdf,DOC_0001_P0003,3,Skew; Equations; Tables,False,False,True,True,True,False,0.912973
3,DOC_0001,biodiversity and conservation.pdf,DOC_0001_P0004,4,Equations; Tables,False,False,False,True,True,False,0.952276
4,DOC_0001,biodiversity and conservation.pdf,DOC_0001_P0005,5,Equations; Tables,False,False,False,True,True,False,0.975166
5,DOC_0001,biodiversity and conservation.pdf,DOC_0001_P0006,6,Skew; Equations; Tables,False,False,True,True,True,False,0.945872
6,DOC_0001,biodiversity and conservation.pdf,DOC_0001_P0007,7,Equations; Tables,False,False,False,True,True,False,0.971822
7,DOC_0001,biodiversity and conservation.pdf,DOC_0001_P0008,8,Equations; Tables,False,False,False,True,True,False,0.920088
8,DOC_0001,biodiversity and conservation.pdf,DOC_0001_P0009,9,Skew; Equations; Tables,False,False,True,True,True,False,0.959456
9,DOC_0001,biodiversity and conservation.pdf,DOC_0001_P0010,10,Skew; Equations; Tables,False,False,True,True,True,False,0.920348


In [14]:
total_pages = len(pages_df)

quality_summary = pd.DataFrame([{

    "total_documents": pages_df["document_id"].nunique(),

    "total_pages": total_pages,

    "total_chunks": len(chunks_df),

    "mean_original_confidence": (
        pages_df["original_confidence"].mean()
    ),

    "mean_preprocessed_confidence": (
        pages_df["preprocessed_confidence"].mean()
    ),

    "pages_with_confidence_improvement": int(
        comparison_df["confidence_improved"].sum()
    ),

    "pages_with_changed_text": int(
        comparison_df["text_changed"].sum()
    ),

    "pages_requiring_review": int(
        pages_df["review_required"].sum()
    ),

    "easy_pages": int(
        (pages_df["processing_category"] == "Easy").sum()
    ),

    "moderate_pages": int(
        (pages_df["processing_category"] == "Moderate").sum()
    ),

    "difficult_pages": int(
        (pages_df["processing_category"] == "Difficult").sum()
    ),

}])

print("QUALITY SUMMARY")
display(quality_summary)

print("\nDIFFICULTY DISTRIBUTION")

difficulty_distribution = (
    pages_df["processing_category"]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="page_count")
)

display(difficulty_distribution)

print("\nPOTENTIAL ERROR SOURCE COUNTS")

error_columns = [
    "handwriting_suspected",
    "low_resolution",
    "skew_detected",
    "equation_detected",
    "table_detected",
    "diagram_suspected"
]

error_summary = pd.DataFrame({

    "error_source": error_columns,

    "page_count": [
        int(error_df[column].sum())
        for column in error_columns
    ]

})

display(error_summary)

QUALITY SUMMARY


,total_documents,total_pages,total_chunks,mean_original_confidence,mean_preprocessed_confidence,pages_with_confidence_improvement,pages_with_changed_text,pages_requiring_review,easy_pages,moderate_pages,difficult_pages
0,9,139,9566,0.911425,0.911479,71,139,0,128,11,0



DIFFICULTY DISTRIBUTION


,category,page_count
0,Easy,128
1,Moderate,11



POTENTIAL ERROR SOURCE COUNTS


,error_source,page_count
0,handwriting_suspected,0
1,low_resolution,0
2,skew_detected,8
3,equation_detected,137
4,table_detected,120
5,diagram_suspected,13


In [15]:
pages_df.to_csv(
    OUTPUT_DIR / "page_ocr_results.csv",
    index=False
)

chunks_df.to_csv(
    OUTPUT_DIR / "chunk_ocr_results.csv",
    index=False
)

comparison_df.to_csv(
    OUTPUT_DIR / "before_after_comparison.csv",
    index=False
)

error_df.to_csv(
    OUTPUT_DIR / "error_analysis.csv",
    index=False
)

quality_summary.to_csv(
    OUTPUT_DIR / "quality_summary.csv",
    index=False
)

difficulty_distribution.to_csv(
    OUTPUT_DIR / "difficulty_distribution.csv",
    index=False
)

error_summary.to_csv(
    OUTPUT_DIR / "error_source_summary.csv",
    index=False
)

with open(
    OUTPUT_DIR / "ocr_outputs.txt",
    "w",
    encoding="utf-8"
) as file:

    for _, row in pages_df.iterrows():

        file.write("\n" + "=" * 80 + "\n")

        file.write(
            f"Document ID: {row['document_id']}\n"
        )

        file.write(
            f"Page ID: {row['page_id']}\n"
        )

        file.write("\nORIGINAL OCR:\n")
        file.write(row["original_text"])

        file.write("\n\nPREPROCESSED OCR:\n")
        file.write(row["preprocessed_text"])

        file.write("\n")

excel_path = OUTPUT_DIR / "surya_ocr_evaluation.xlsx"

with pd.ExcelWriter(
    excel_path,
    engine="openpyxl"
) as writer:

    pages_df.to_excel(
        writer,
        sheet_name="Page OCR Results",
        index=False
    )

    chunks_df.to_excel(
        writer,
        sheet_name="Chunk OCR Results",
        index=False
    )

    comparison_df.to_excel(
        writer,
        sheet_name="Before After",
        index=False
    )

    error_df.to_excel(
        writer,
        sheet_name="Error Analysis",
        index=False
    )

    quality_summary.to_excel(
        writer,
        sheet_name="Quality Summary",
        index=False
    )

    difficulty_distribution.to_excel(
        writer,
        sheet_name="Difficulty",
        index=False
    )

    error_summary.to_excel(
        writer,
        sheet_name="Error Sources",
        index=False
    )


print("All deliverables exported.")
print("Output directory:", OUTPUT_DIR)

All deliverables exported.
Output directory: /content/handwritten_ocr/outputs


In [17]:
import shutil
from google.colab import files

zip_path = "/content/ocr_deliverables.zip"

shutil.make_archive(
    "/content/ocr_deliverables",
    "zip",
    OUTPUT_DIR
)

files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>